# Prepping Data Challenge - Week 21

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
customer_spending_df = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_21_Loyalty_Programm.csv')

In [3]:
customer_spending_df.head()

,First Name,Last Name,Gender,Receipt Number,Date,Online,In Person,Sale Total
0,Emeline,Woollard,Female,2578226121,27/12/2023,No,Yes,250.54
1,Wallace,Slyde,Male,6574211891,6/8/2023,No,Yes,363.43
2,Chrissy,MacGiany,Male,7742449501,19/4/2023,No,Yes,487.25
3,Jacques,Brauns,Male,3112564839,25/12/2023,No,Yes,195.67
4,Joannes,Cabrera,Female,8018081174,20/7/2023,No,Yes,95.10


In [4]:
customer_spending_df.shape

(999, 8)

## Challenge

Create a loyalty point system, with 1 point for every 50€ spent. Categorize into groups of 
- MegaByte (7+ Points),
- Byte (5-6 Points)
- No Byte (>= 4 Points)

Break down percentage of customers in each category by `Gender`

### Check if all customers are unique

In [5]:
customer_spending_df['Name'] = customer_spending_df['First Name'] + ' ' + customer_spending_df['Last Name']

In [6]:
customer_spending_df['Name'].nunique()

999

Number of unique names matches number of rows. No deduplication required.

### Create Loyalty system

In [7]:
customer_spending_df['Loyalty Points'] = (customer_spending_df['Sale Total'] // 50).astype(int)

In [8]:
customer_spending_df['Loyalty Tier'] = np.where(customer_spending_df['Loyalty Points'] >= 5, 'Byte', np.nan)
customer_spending_df['Loyalty Tier'] = np.where(customer_spending_df['Loyalty Points'] >= 7, 'Mega Byte', customer_spending_df['Loyalty Tier'])
customer_spending_df['Loyalty Tier'] = np.where(customer_spending_df['Loyalty Points'] < 5, 'No Byte', customer_spending_df['Loyalty Tier'])

In [9]:
customer_spending_df['Loyalty Tier'].value_counts()

Loyalty Tier
No Byte      490
Mega Byte    308
Byte         201
Name: count, dtype: int64

### Categorize by Gender

In [10]:
loyalty_by_customer_df = customer_spending_df.groupby('Gender')['Loyalty Tier'].value_counts().unstack().T

loyalty_by_customer_df = loyalty_by_customer_df.loc[['No Byte', 'Byte', 'Mega Byte']] # Rearrange Rows

In [11]:
all_customers_in_tier = loyalty_by_customer_df['Female'] + loyalty_by_customer_df['Male']

In [12]:
loyalty_by_customer_df['Male'] = (loyalty_by_customer_df['Male'] / all_customers_in_tier).round(3) * 100
loyalty_by_customer_df['Female'] = (loyalty_by_customer_df['Female'] / all_customers_in_tier).round(3) * 100

In [13]:
loyalty_by_customer_df

Gender,Female,Male
Loyalty Tier,,
No Byte,49.2,50.8
Byte,55.7,44.3
Mega Byte,47.1,52.9


## Save Results

In [14]:
loyalty_by_customer_df.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/21_loyalty_tiers_by_gender.csv')